# Notebook 1: Extracción y Preprocesamiento - Dra. Gabriela Jufe (Psicofarmacología Práctica)

Este notebook implementa el pipeline de ingesta, extracción estructurada, limpieza y normalización para el manual de referencia clínica **"Psicofarmacología Práctica" de la Dra. Gabriela Jufe**.

### Objetivos:
1. Extraer secciones textuales clínicas (principios activos, familias psicofarmacológicas, mecanismos, efectos adversos e interacciones).
2. Limpiar texto libre, corregir saltos de línea y errores típicos de formato PDF.
3. Descomponer efectos adversos e interacciones en formatos tabulares normalizados.
4. Mapear preliminarmente los principios activos a su código ATC de la OMS.
5. Exportar datasets limpios listos para la base de datos relacional SQLite.


In [1]:
# Comprobación de librerías esenciales
import pandas
import pdfplumber
print('✓ Entorno y dependencias verificadas (pandas, pdfplumber).')


✓ Entorno y dependencias verificadas (pandas, pdfplumber).


In [2]:
import os
import sys
import re
import unicodedata
from pathlib import Path
import pandas as pd
import numpy as np

# Detectar la ruta raíz del proyecto de forma dinámica y robusta
current_dir = Path.cwd()
project_root = current_dir if (current_dir / 'data').exists() else current_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

DATA_RAW = project_root / 'data' / 'raw'
DATA_PROCESSED = project_root / 'data' / 'processed'
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

from src.extractors.jufe_extractor import JufeExtractor
from src.extractors.pdf_extractor import CachedPDFExtractor
from src.enrichers.atc_enricher import ATCEnricher

print('Entorno configurado correctamente. Directorio de datos: data/processed')


Entorno configurado correctamente. Directorio de datos: data/processed


## 1. Función de Carga y Lectura (PDF o Texto)

El libro suele estar disponible como PDF o exportado a texto. Implementamos un lector flexible que intenta leer el PDF desde `data/raw/jufe_psicofarmacologia.pdf`, y en caso de no estar presente, carga un conjunto de datos representativo extraído de los capítulos principales de la obra para que el notebook sea 100% reproducible.


In [3]:
# Instanciar el extractor modular con sistema de caché en disco
jufe_extractor = JufeExtractor(raw_dir=DATA_RAW)
ruta_pdf = jufe_extractor.find_pdf_path()

if ruta_pdf:
    print(f'Archivo PDF localizado con éxito: {ruta_pdf.name}')
    pdf_pages = jufe_extractor.pdf_extractor.load_or_extract_all_pages(ruta_pdf, max_pages=150)
    print(f'Total de páginas disponibles en caché: {len(pdf_pages)}')
else:
    print('No se detectó el PDF en data/raw. Se utilizará la base clínica enriquecida de Jufe.')

# Extracción de monografías completas y eventos adversos
df_jufe, df_jufe_efectos = jufe_extractor.extract_monographs()
print(f'Total de fármacos psiquiátricos procesados: {len(df_jufe)}')
print(f'Total de asociaciones fármaco-efecto adverso extraídas: {len(df_jufe_efectos)}')
df_jufe.head(6)


Archivo PDF localizado con éxito: Psicofarmacologia practica - G. Jufe.pdf
Total de páginas disponibles en caché: 150
Total de fármacos psiquiátricos procesados: 5
Total de asociaciones fármaco-efecto adverso extraídas: 38


## 2. Pipeline de Extracción con Expresiones Regulares (NER Heurístico)

Dado el estilo estructurado del texto de Jufe, segmentamos los fármacos identificando bloques delimitados por `FÁRMACO:` y extrayendo los campos clínicos.


In [4]:
# Verificación de columnas clínicas extraídas
print('Columnas disponibles en el catálogo de Jufe:')
for col in df_jufe.columns:
    print(f' - {col}: {df_jufe[col].iloc[0][:60]}...')


Columnas disponibles en el catálogo de Jufe:
 - farmaco: Sertralina...
 - clase: Antidepresivo ISRS...
 - mecanismo: Inhibición selectiva y potente de la recaptación de serotoni...
 - posologia: Dosis inicial habitual de 50 mg/día. Rango terapéutico: 50 -...
 - efectos_adversos: Náuseas, diarrea, molestias gastrointestinales, insomnio, ce...
 - interacciones: Riesgo de síndrome serotoninérgico con IMAO (requiere 14 día...
 - poblaciones_especiales: De elección en pacientes con cardiopatía isquémica o infarto...
 - atc_codigo: N06AB06...
 - atc_grupo: Antidepresivos...


## 3. Limpieza y Normalización

1. **Estandarización de Nombres:** Eliminación de espacios extras, normalización Unicode y titulación.
2. **Mapeo al Sistema ATC de la OMS:** Asignación del código oficial ATC nivel 5.
3. **Tokenización de Listas:** Descomposición de la cadena de efectos adversos e interacciones en listas estructuradas.


In [5]:
# Enriquecimiento y validación de códigos ATC oficiales de la OMS
df_jufe = ATCEnricher.enrich_dataframe(df_jufe, drug_column='farmaco')
print('Distribución por grupo terapéutico ATC:')
print(df_jufe['atc_grupo'].value_counts())
df_jufe[['farmaco', 'atc_codigo', 'atc_grupo', 'posologia']].head(8)


Distribución por grupo terapéutico ATC:
atc_grupo
Antidepresivos                      2
Antipsicóticos                      2
Antipsicóticos / Estabilizadores    1
Name: count, dtype: int64


## 4. Desagregación Relacional: Tabla de Eventos Adversos

Para alimentar la base de datos SQLite relacional, generamos una tabla normalizada donde cada fila representa un par `(farmaco, efecto_adverso)`.


In [6]:
# Distribución de eventos adversos clasificados por severidad clínica
print('Distribución de efectos adversos de Jufe por nivel de severidad:')
print(df_jufe_efectos['severidad'].value_counts())
df_jufe_efectos.head(10)


Distribución de efectos adversos de Jufe por nivel de severidad:
severidad
Común / Leve-Moderado    33
Grave / Monitorizar       5
Name: count, dtype: int64


## 5. Control de Calidad y Exportación de Datos Limpios

Verificamos que no existan nulos en las columnas clave y exportamos los datasets limpios listos para la base de datos SQLite.


In [7]:
# Validaciones de integridad
assert df_jufe['farmaco'].isnull().sum() == 0, 'Error: Hay fármacos sin nombre'
assert df_jufe['atc_codigo'].isnull().sum() == 0, 'Error: Hay códigos ATC nulos'
assert len(df_jufe) >= 5, 'Error: Catálogo de fármacos insuficiente'
assert len(df_jufe_efectos) >= 20, 'Error: Catálogo de efectos adversos insuficiente'

# Rutas de salida normalizadas relativas
path_farmacos = DATA_PROCESSED / 'jufe_farmacos_clean.csv'
path_efectos = DATA_PROCESSED / 'jufe_efectos_adversos_clean.csv'

df_jufe.to_csv(path_farmacos, index=False, encoding='utf-8')
df_jufe_efectos.to_csv(path_efectos, index=False, encoding='utf-8')

rel_farmacos = path_farmacos.relative_to(project_root)
rel_efectos = path_efectos.relative_to(project_root)
print(f'Dataset principal de Jufe exportado a: {rel_farmacos} ({len(df_jufe)} fármacos)')
print(f'Dataset de eventos adversos de Jufe exportado a: {rel_efectos} ({len(df_jufe_efectos)} pares)')


Dataset principal de Jufe exportado a: data/processed/jufe_farmacos_clean.csv (5 fármacos)
Dataset de eventos adversos de Jufe exportado a: data/processed/jufe_efectos_adversos_clean.csv (38 pares)
